# NeuroXVocal — Evaluation on Linux Server
**Standalone evaluation notebook** — run this after training is complete.

- Reads training results from `/mnt/ssd2/Ali/results/`
- Auto-selects best fold from `training.log`
- Evaluates on 71 test patients
- Tracks VRAM & timing per pipeline

## 1. Check GPU

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU             : {props.name}')
    print(f'Total VRAM      : {props.total_memory / 1024**3:.1f} GB')

## 2. Configure Paths

In [ ]:
import os, re

# ── CONFIGURE THESE ─────────────────────────────────────────────────
REPO_DIR      = os.path.expanduser('~/NeuroXVocal')
RESULTS_DIR   = '/mnt/ssd2/Ali/results'
TEST_WAV_DIR  = '/mnt/ssd2/Ali/ADReSSo/diagnosis-test/diagnosis/test-dist/audio'
OUTPUT_CSV    = os.path.join(RESULTS_DIR, 'predictions.csv')
MONITOR_CSV   = os.path.join(RESULTS_DIR, 'vram_monitor.csv')
LABELS_CSV    = None  # set to CSV with (patient_id, label) if you have ground truth
# ────────────────────────────────────────────────────────────────────

# Auto-select best fold from training.log
LOG_PATH = os.path.join(RESULTS_DIR, 'training.log')
pattern  = re.compile(r'Fold (\d+), Epoch \d+, Train Loss: [\d.]+, Val Loss: ([\d.]+)')
folds    = {}
with open(LOG_PATH) as f:
    for line in f:
        m = pattern.search(line)
        if m:
            fold, vl = int(m.group(1)), float(m.group(2))
            if fold not in folds or vl < folds[fold]:
                folds[fold] = vl

print('Val loss per fold:')
for fold, vl in sorted(folds.items()):
    print(f'  Fold {fold}: {vl:.4f}')

best_fold  = min(folds, key=folds.get)
MODEL_PATH = os.path.join(RESULTS_DIR, f'model_fold{best_fold}_best.pth')
print(f'\nBest fold   : {best_fold}  (val loss {folds[best_fold]:.4f})')
print(f'Model path  : {MODEL_PATH}')
print(f'Model exists: {os.path.exists(MODEL_PATH)}')

from pathlib import Path
wavs = list(Path(TEST_WAV_DIR).glob('*.wav'))
print(f'\nTest audio files: {len(wavs)}')

## 3. VRAM Monitor Class

In [ ]:
import time, torch

class VRAMMonitor:
    def __init__(self, device):
        self.device       = device
        self.dtype        = device.type
        self._global_peak = 0.0
        self.load_records = []

    def current_mb(self):
        return torch.cuda.memory_allocated(self.device) / 1024**2 if self.dtype == 'cuda' else 0.0

    def peak_mb(self):
        return torch.cuda.max_memory_allocated(self.device) / 1024**2 if self.dtype == 'cuda' else 0.0

    def reset_peak(self):
        if self.dtype == 'cuda': torch.cuda.reset_peak_memory_stats(self.device)

    def sync(self):
        if self.dtype == 'cuda': torch.cuda.synchronize(self.device)

    def measure(self, label):
        return _Stage(self, label)

    def record_load(self, name, fn):
        self.sync()
        before = self.current_mb()
        obj    = fn()
        self.sync()
        after  = self.current_mb()
        delta  = after - before
        self.load_records.append({'model': name, 'before_mb': round(before, 1),
                                   'after_mb': round(after, 1), 'delta_mb': round(delta, 1)})
        print(f'  Loaded {name:<26}  before={before:6.1f} MB  after={after:6.1f} MB  delta=+{delta:.1f} MB')
        return obj

    def update_global_peak(self, v):
        if v > self._global_peak: self._global_peak = v

    def global_peak_mb(self):
        return self._global_peak


class _Stage:
    def __init__(self, mon, label):
        self.mon   = mon
        self.label = label
        self.vram_before = self.vram_after = self.peak = self.elapsed = 0.0

    def __enter__(self):
        self.mon.sync()
        self.mon.reset_peak()
        self.vram_before = self.mon.current_mb()
        self._t0         = time.perf_counter()
        return self

    def __exit__(self, *_):
        self.mon.sync()
        self.elapsed    = time.perf_counter() - self._t0
        self.vram_after = self.mon.current_mb()
        self.peak       = self.mon.peak_mb()
        self.mon.update_global_peak(self.peak)

print('VRAMMonitor ready.')

## 4. Pre-Extract All Test Audio Features
Runs extraction once for all 71 patients — avoids O(n²) bottleneck.

In [ ]:
import sys, time, joblib
import pandas as pd, numpy as np

SCALER_FEAT   = os.path.join(REPO_DIR, 'src/inference/scaler_params_audio_features.pkl')
FEAT_SCRIPT   = os.path.join(REPO_DIR, 'src/data_extraction/extract_audio_features.py')
TEST_FEAT_CSV = os.path.join(RESULTS_DIR, 'test_features_raw.csv')
DROP_COLS     = ['jitter_local', 'shimmer_local', 'formant_1_mean', 'formant_1_std',
                 'formant_2_mean', 'formant_2_std', 'formant_3_mean', 'formant_3_std', 'class']

print('Pre-extracting test audio features...')
t0 = time.perf_counter()
!{sys.executable} {FEAT_SCRIPT} {TEST_WAV_DIR} --output_csv {TEST_FEAT_CSV}
print(f'Done in {time.perf_counter()-t0:.1f}s')

df_feat_raw = pd.read_csv(TEST_FEAT_CSV)
feat_ids    = df_feat_raw['patient_id'].astype(str)
feat_data   = df_feat_raw.drop(columns=['patient_id'] + [c for c in DROP_COLS if c in df_feat_raw.columns])
feat_data   = feat_data.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
feat_scaled = joblib.load(SCALER_FEAT).transform(feat_data)
df_feat     = pd.DataFrame(feat_scaled, columns=feat_data.columns)
df_feat.insert(0, 'patient_id', feat_ids.values)
print(f'Features ready: {df_feat.shape[0]} patients x {df_feat.shape[1]-1} features')

## 5. Load All Models (with VRAM tracking)

In [ ]:
import sys, re, warnings, joblib
import whisper, soundfile as sf
import pandas as pd, numpy as np
from pathlib import Path
from math import gcd
from scipy.signal import resample_poly
from transformers import AutoTokenizer, Wav2Vec2Model, Wav2Vec2Processor

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn')

sys.path.insert(0, os.path.join(REPO_DIR, 'src/train'))
from models import NeuroXVocal

SCALER_EMB = os.path.join(REPO_DIR, 'src/inference/scaler_params_audio_emb.pkl')
TEXT_MODEL = 'microsoft/deberta-v3-base'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mon    = VRAMMonitor(device)
print(f'Device: {device}\n')

wmodel    = mon.record_load('Whisper base',
                lambda: whisper.load_model('base'))
tokenizer = mon.record_load('DeBERTa tokenizer',
                lambda: AutoTokenizer.from_pretrained(TEXT_MODEL))
processor = mon.record_load('Wav2Vec2 processor',
                lambda: Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h'))
emodel    = mon.record_load('Wav2Vec2 model',
                lambda: Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h').to(device).eval())

def _load_clf():
    clf = NeuroXVocal(num_audio_features=47, num_embedding_features=768, text_embedding_model=TEXT_MODEL)
    sd  = torch.load(MODEL_PATH, map_location=device)
    if 'module.' in list(sd.keys())[0]:
        from collections import OrderedDict
        sd = OrderedDict((k.replace('module.', ''), v) for k, v in sd.items())
    clf.load_state_dict(sd)
    return clf.to(device).eval()

clf = mon.record_load('NeuroXVocal (DeBERTa)', _load_clf)

print('\n── Model Load Summary ──────────────────────────────────────────')
print(f'  {"Model":<28}  {"Before":>9}  {"After":>9}  {"Delta":>9}')
print('  ' + '-'*58)
for r in mon.load_records:
    print(f'  {r["model"]:<28}  {r["before_mb"]:>8.1f}M  {r["after_mb"]:>8.1f}M  +{r["delta_mb"]:>7.1f}M')

## 6. Run Evaluation
3 pipelines tracked per patient: Whisper → Wav2Vec2 → NeuroXVocal

In [ ]:
from tqdm import tqdm

def transcribe(wav_path):
    r = wmodel.transcribe(str(wav_path), fp16=torch.cuda.is_available())
    t = r['text'].lower()
    return re.sub(r'[^a-zA-Z0-9\s.,!?]', '', ' '.join(t.split()))

def get_features(patient_id):
    row = df_feat[df_feat['patient_id'].astype(str) == str(patient_id)]
    if row.empty:
        raise ValueError(f'No features found for {patient_id}')
    return row

def get_embeddings(wav_path):
    speech, sr = sf.read(str(wav_path), always_2d=True)
    speech     = speech.mean(axis=1).astype(np.float32)
    if sr != 16000:
        g      = gcd(sr, 16000)
        speech = resample_poly(speech, 16000 // g, sr // g).astype(np.float32)
    inp = processor(speech, sampling_rate=16000, return_tensors='pt', padding=True)
    inp = {k: v.to(device) for k, v in inp.items()}
    with torch.no_grad():
        emb = emodel(**inp).last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return joblib.load(SCALER_EMB).transform(emb.reshape(1, -1)).flatten()

def predict(text, feat_row, emb):
    tok  = tokenizer(text, padding='max_length', truncation=True, max_length=512, return_tensors='pt')
    tok  = {k: v.to(device) for k, v in tok.items()}
    af   = torch.tensor(feat_row.drop(columns=['patient_id']).iloc[0].values.astype(float),
                        dtype=torch.float32).unsqueeze(0).to(device)
    embt = torch.tensor(emb, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        conf = torch.sigmoid(clf(tok, af, embt)).item()
    return (1 if conf > 0.5 else 0), conf

# ── Main loop ────────────────────────────────────────────────────────
wav_files       = sorted(Path(TEST_WAV_DIR).glob('*.wav'))
results         = []
monitor_records = []

for wav in tqdm(wav_files, desc='Patients'):
    pid = wav.stem
    try:
        with mon.measure('1. Whisper transcription') as s1:
            text = transcribe(wav)
        monitor_records.append({'patient': pid, 'pipeline': '1. Whisper transcription',
                                'vram_before_mb': round(s1.vram_before, 1), 'vram_peak_mb': round(s1.peak, 1),
                                'vram_after_mb': round(s1.vram_after, 1), 'time_sec': round(s1.elapsed, 3)})

        t0       = time.perf_counter()
        feat_row = get_features(pid)
        monitor_records.append({'patient': pid, 'pipeline': '2a. AudioFeat (lookup)',
                                'vram_before_mb': None, 'vram_peak_mb': None,
                                'vram_after_mb': None, 'time_sec': round(time.perf_counter() - t0, 4)})

        with mon.measure('2b. Wav2Vec2 embedding') as s2:
            emb = get_embeddings(wav)
        monitor_records.append({'patient': pid, 'pipeline': '2b. Wav2Vec2 embedding',
                                'vram_before_mb': round(s2.vram_before, 1), 'vram_peak_mb': round(s2.peak, 1),
                                'vram_after_mb': round(s2.vram_after, 1), 'time_sec': round(s2.elapsed, 3)})

        with mon.measure('3. NeuroXVocal classify') as s3:
            pred, conf = predict(text, feat_row, emb)
        monitor_records.append({'patient': pid, 'pipeline': '3. NeuroXVocal classify',
                                'vram_before_mb': round(s3.vram_before, 1), 'vram_peak_mb': round(s3.peak, 1),
                                'vram_after_mb': round(s3.vram_after, 1), 'time_sec': round(s3.elapsed, 3)})

        results.append({'patient_id': pid, 'prediction': pred,
                        'label': 'AD' if pred == 1 else 'CN',
                        'confidence': round(conf, 4), 'transcription': text[:120]})

    except Exception as e:
        print(f'ERROR {pid}: {e}')
        results.append({'patient_id': pid, 'prediction': None,
                        'label': 'ERROR', 'confidence': None, 'transcription': str(e)})

df_results = pd.DataFrame(results)
df_monitor = pd.DataFrame(monitor_records)
df_results.to_csv(OUTPUT_CSV,  index=False)
df_monitor.to_csv(MONITOR_CSV, index=False)
print(f'\nPredictions saved → {OUTPUT_CSV}')
print(f'VRAM log saved    → {MONITOR_CSV}')
df_results[['patient_id', 'label', 'confidence']]

## 7. Prediction Summary

In [ ]:
valid = df_results.dropna(subset=['prediction'])
print(f'Total processed : {len(valid)} / {len(df_results)}')
print(f'Predicted AD    : {(valid["prediction"]==1).sum()}')
print(f'Predicted CN    : {(valid["prediction"]==0).sum()}')
print(f'\nConfidence stats:')
print(valid['confidence'].describe().round(4))

## 8. VRAM & Timing Report

In [ ]:
import pandas as pd

df_load = pd.DataFrame(mon.load_records)
print('MODEL LOAD — VRAM COST')
print('=' * 62)
print(df_load.to_string(index=False))

tracked = df_monitor[df_monitor['vram_peak_mb'].notna()].copy()
summary = tracked.groupby('pipeline').agg(
    peak_max  = ('vram_peak_mb', 'max'),
    peak_min  = ('vram_peak_mb', 'min'),
    peak_mean = ('vram_peak_mb', 'mean'),
    time_mean = ('time_sec',     'mean'),
    time_total= ('time_sec',     'sum'),
).round(2)

print('\nVRAM & TIMING SUMMARY — per pipeline')
print('=' * 72)
print(f'{"Pipeline":<28}  {"Peak max":>9}  {"Peak min":>9}  {"Peak avg":>9}  {"Avg time":>9}')
print('-' * 72)
for pipe, row in summary.iterrows():
    print(f'{pipe:<28}  {row.peak_max:>8.1f}M  {row.peak_min:>8.1f}M  {row.peak_mean:>8.1f}M  {row.time_mean:>8.2f}s')
print('=' * 72)
print(f'Global peak VRAM : {mon.global_peak_mb():.1f} MB')
print(f'Global min  VRAM : {tracked["vram_peak_mb"].min():.1f} MB')

## 9. Accuracy vs Ground Truth (optional)
Set `LABELS_CSV` in Section 2 to a CSV with columns `patient_id` and `label` (1=AD, 0=CN).

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

if LABELS_CSV is not None:
    gt     = pd.read_csv(LABELS_CSV)
    merged = df_results.merge(gt, on='patient_id', suffixes=('_pred', '_true'))
    merged = merged.dropna(subset=['prediction'])
    y_true = merged['label_true'].astype(int).values
    y_pred = merged['prediction'].astype(int).values
    acc    = accuracy_score(y_true, y_pred)
    print(f'Test Accuracy: {acc*100:.2f}%  ({int(acc*len(y_true))}/{len(y_true)})\n')
    print(classification_report(y_true, y_pred, target_names=['CN', 'AD']))
    print('Confusion Matrix:')
    print(pd.DataFrame(confusion_matrix(y_true, y_pred),
                       index=['True CN', 'True AD'], columns=['Pred CN', 'Pred AD']))
else:
    print('LABELS_CSV not set. Provide ground truth CSV to compute accuracy.')